# Phase 2: planner baselines (greedy and self-consistency)

Runs a small open LLM on Blocksworld problems, checks every plan with the Python simulator,
and saves all samples. Use a **GPU runtime** (Colab: Runtime > Change runtime type > T4 GPU).

Before running: push your latest code to GitHub (this notebook clones the public repo).


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import os
REPO = "https://github.com/Manidip-Debnath/llm-planning-process-verifier.git"
if not os.path.exists("llm-planning-process-verifier"):
    !git clone {REPO}
%cd llm-planning-process-verifier
!git log --oneline | head -3

## 1. Generate the problems (same seed as on your PC, so the problems are identical)

In [ ]:
!python data/gen_blocksworld.py --sizes 3 4 5 6 --count 25 --seed 0 --out data/blocksworld/problems
!ls data/blocksworld/problems | wc -l

## 2. Smoke test (3 problems, small model). Do this first.

In [ ]:
!python planners/run_baselines.py --backend hf --model Qwen/Qwen2.5-1.5B-Instruct \
    --problems data/blocksworld/problems --limit 3 --n-samples 8 --out data/runs/smoke.jsonl

In [ ]:
# Look at what the model actually wrote for the first problem
import json
r = json.loads(open("data/runs/smoke.jsonl").readline())
print("GREEDY OUTPUT:\n", r["greedy"]["text"])
print("parsed plan:", r["greedy"]["plan"])
print("valid:", r["greedy"]["valid"], "| reason:", r["greedy"]["reason"])

## 3. Full baseline run

Pick the planner model. Start with 1.5B; switch to 3B when the smoke test looks sane.
If the greedy validity is near 0% or near 100%, change the block sizes in step 1 (see docs/06_PHASE2_RUN_GUIDE.md).


In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"   # or "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "data/runs/" + MODEL.split("/")[-1] + ".jsonl"
!python planners/run_baselines.py --backend hf --model {MODEL} \
    --problems data/blocksworld/problems --n-samples 32 --out {OUT}

## 4. Summarize (optimality gap needs solutions.jsonl, which you can compute on your PC and skip here)

In [ ]:
!python eval/summarize_baselines.py --results {OUT} --out results/baselines_{MODEL.split("/")[-1]}.md

## 5. Download the results to your PC

In [ ]:
!zip -q -r results_download.zip data/runs results
from google.colab import files
files.download("results_download.zip")